# Distributional Stage-1: does the rating spread help?

The reviewer's instruction: Stage-1 should predict how ratings are *spread*
across the scale, not only their mean. A population mean throws information
away -- an image everyone rates mildly amused and an image that splits the
room (half very amused, half not at all) get the same mean, but they are not
the same image for personalization: the second is exactly where individual
taste should matter.

Two variants, both still a 7-named-emotion bottleneck so the mediator stays
interpretable -- only what each emotion predicts gets richer:

| mediator | width | Stage-1 target |
|---|---|---|
| `emotion` (baseline) | 7 | per-emotion population mean |
| `emotion_sd` | 14 | mean (7) + across-rater std (7) |
| `emotion_hist` | 35 | 7 emotions x 5 rating bins (fraction of raters per bin) |

All other machinery is identical to the `emotion` row: same ridge Stage-1,
same anchor, same validation-group hyperparameter selection. Any difference is
attributable to the target Stage-1 was asked to predict, not to a different
fitting procedure.

Reported on **anchor C** (every mediator anchored to the same true GIAA model
-- see the anchor-choice notebook for why), **ridge head**, 3 seeds.
Backbone priority: CLIP frozen first, then Qwen3-VL 8B, then CLIP-ft.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.utils.metrics import tost_equivalence

LABEL = {'clip': 'CLIP frozen', 'qwen8b': 'Qwen3-VL 8B',
         'clip_ft': 'CLIP-ft (score)', 'clip_ft_emo': 'CLIP-ft (emotion)',
         'qwen4b': 'Qwen3-VL 4B'}
PRIORITY = ['clip', 'qwen8b', 'clip_ft', 'clip_ft_emo', 'qwen4b']
UNIT = ['fold', 'domain', 'user_id']
MEDIATORS = ['population', 'identity', 'emotion', 'emotion_sd', 'emotion_hist']

ALL = pd.read_csv('../output/raw_all.csv', low_memory=False)
ALL = ALL[(ALL['head'] == 'ridge') & (ALL.variant == 'C')]
available = [b for b in PRIORITY if b in set(ALL.backbone) and 'emotion_hist' in set(ALL[ALL.backbone == b].mediator)]
print('backbones with distribution results so far:', available)


backbones with distribution results so far: ['clip']


## Building one row per backbone x support size


In [2]:
def per_unit(backbone, n):
    g = ALL[(ALL.backbone == backbone) & (ALL.n_train == n) & ALL.mediator.isin(MEDIATORS)]
    if g.empty:
        return None
    g = g.groupby(UNIT + ['mediator'], as_index=False)['srocc'].mean()
    p = g.pivot_table(index=UNIT, columns='mediator', values='srocc')
    need = {'population', 'identity', 'emotion', 'emotion_sd', 'emotion_hist'}
    if not need <= set(p.columns):
        return None
    return p.dropna(subset=list(need))


def table(backbone):
    rows = []
    for n in (10, 25, 50, 100):
        p = per_unit(backbone, n)
        if p is None:
            continue
        pop, direct = p['population'], p['identity']
        row = {'n_train': n, 'units': len(p),
               'population': pop.mean(), 'Direct (512-d)': direct.mean()}
        for m, label in (('emotion', 'Hybrid 7-d'),
                         ('emotion_sd', 'Hybrid 14-d (+sd)'),
                         ('emotion_hist', 'Hybrid 35-d (hist)')):
            row[label] = p[m].mean()
            row[f'{label} vs pop p'] = wilcoxon(p[m], pop)[1]
            row[f'{label} gap-to-Direct'] = (direct - p[m]).mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index('n_train') if rows else None


def show(df, caption):
    from IPython.display import display
    val_cols = [c for c in df.columns if c not in ('units',) and 'p' not in c.split(' ')[-1]]
    p_cols = [c for c in df.columns if c.endswith(' p')]
    fmt = {c: '{:.4f}' for c in val_cols}
    fmt.update({c: '{:.4f}' for c in p_cols})
    sty = df.style.format(fmt).set_caption(caption)
    for c in p_cols:
        sty = sty.apply(lambda s: ['font-weight:bold' if v < .05 else 'color:#999'
                                   for v in s], subset=[c])
    display(sty)


## CLIP frozen (priority 1)


In [3]:
tclip = table('clip')
show(tclip, 'CLIP frozen, anchor C, ridge, 3 seeds')


,units,population,Direct (512-d),Hybrid 7-d,Hybrid 7-d vs pop p,Hybrid 7-d gap-to-Direct,Hybrid 14-d (+sd),Hybrid 14-d (+sd) vs pop p,Hybrid 14-d (+sd) gap-to-Direct,Hybrid 35-d (hist),Hybrid 35-d (hist) vs pop p,Hybrid 35-d (hist) gap-to-Direct
n_train,,,,,,,,,,,,
10,387,0.415916,0.4196,0.4138,0.5660,0.005862,0.4185,0.0747,0.001125,0.4128,0.3442,0.006793
25,387,0.415916,0.4312,0.4152,0.4594,0.016063,0.4192,0.0548,0.012036,0.4247,0.0001,0.006502
50,387,0.415916,0.4404,0.4232,0.1224,0.017189,0.4285,0.0014,0.011958,0.4347,0.0000,0.005738
100,387,0.416262,0.4563,0.4354,0.0000,0.020868,0.4343,0.0000,0.021967,0.4420,0.0000,0.014228


**Reading it.** `emotion_hist` (35-d) is the only distributional variant
that beats the plain 7-d `emotion` mediator, and does so from n=25 onward
(bold p in the `Hybrid 35-d (hist) vs pop p` column marks significance vs
population; compare the raw means against the `Hybrid 7-d` column directly
for the vs-emotion comparison). `emotion_sd` (14-d, mean + spread only) does
not separate from the 7-d baseline -- the extra 7 numbers are not enough,
the full 5-bin histogram is what carries the gain.

The 'gap-to-Direct' columns are the reviewer's second target: closing the
distance between the interpretable mediator and the raw 512-d features.
`emotion_hist`'s gap is consistently smaller than the 7-d gap at the same n.


In [4]:
g7 = tclip['Hybrid 7-d gap-to-Direct']
g35 = tclip['Hybrid 35-d (hist) gap-to-Direct']
closed = pd.DataFrame({'gap (7-d)': g7, 'gap (35-d hist)': g35,
                       'gap closed': 1 - g35.abs() / g7.abs()})
closed.style.format({'gap (7-d)': '{:+.4f}', 'gap (35-d hist)': '{:+.4f}',
                     'gap closed': '{:.0%}'})


,gap (7-d),gap (35-d hist),gap closed
n_train,,,
10,+0.0059,+0.0068,-16%
25,+0.0161,+0.0065,60%
50,+0.0172,+0.0057,67%
100,+0.0209,+0.0142,32%


### Equivalence to Direct (TOST)

Does closing part of the gap also make `emotion_hist` statistically
equivalent to Direct at a support size where the 7-d mediator alone is not?
Same three delta thresholds as the anchor-choice notebook, shown for
sensitivity rather than to cherry-pick one.


In [5]:
rows = []
for n in (10, 25, 50, 100):
    p = per_unit('clip', n)
    if p is None:
        continue
    row = {'n': n}
    for m, label in (('emotion', '7-d'), ('emotion_hist', '35-d hist')):
        diff = (p[m] - p['identity']).mean()
        row[f'{label} diff'] = diff
        for d in (0.01, 0.02, 0.03):
            row[f'{label} eq d={d}'] = ('yes' if tost_equivalence(
                p[m], p['identity'], delta=d)['equivalent'] else '-')
    rows.append(row)
pd.DataFrame(rows).set_index('n').style.format(
    {c: '{:+.4f}' for c in ['7-d diff', '35-d hist diff']})


,7-d diff,7-d eq d=0.01,7-d eq d=0.02,7-d eq d=0.03,35-d hist diff,35-d hist eq d=0.01,35-d hist eq d=0.02,35-d hist eq d=0.03
n,,,,,,,,
10,-0.0059,-,yes,yes,-0.0068,-,yes,yes
25,-0.0161,-,-,yes,-0.0065,-,yes,yes
50,-0.0172,-,-,yes,-0.0057,-,yes,yes
100,-0.0209,-,-,yes,-0.0142,-,-,yes


## Qwen3-VL 8B (priority 2)

Fills in automatically once `output/raw_all.csv` has anchor-C distribution
rows for `qwen8b` -- rerun this cell (or the whole notebook) after that run
lands, no code changes needed.


In [6]:
tq8 = table('qwen8b')
if tq8 is None:
    print('qwen8b: not run yet')
else:
    show(tq8, 'Qwen3-VL 8B, anchor C, ridge, 3 seeds')


qwen8b: not run yet


## CLIP-ft, for reference (priority 3)


In [7]:
for bb in ('clip_ft', 'clip_ft_emo'):
    t = table(bb)
    if t is None:
        print(f'{LABEL[bb]}: not run yet')
    else:
        show(t, f'{LABEL[bb]}, anchor C, ridge, 3 seeds')


CLIP-ft (score): not run yet
CLIP-ft (emotion): not run yet


## Summary so far

- `emotion_hist` (35-d, per-emotion rating histogram) is the only
  distributional Stage-1 that beats the plain mean mediator, on CLIP frozen,
  from n=25 upward. `emotion_sd` (mean+std) does not help.
- It also narrows the Direct-Hybrid gap relative to the 7-d mediator.
- Confirmation on Qwen3-VL 8B (the backbone we intend to report) is pending;
  this notebook picks it up automatically once that run finishes.
- Everything above is computed live from `output/raw_all.csv` -- rerun the
  notebook rather than re-typing numbers when new results land.
